# Bayesian Workflow for Synthetic Long COVID Symptom Durations

This Colab notebook is the runnable companion to the Bayesian Playground website. It uses a realistic synthetic dataset so the whole workflow is reproducible without sharing patient-level data.

The analysis follows a defensible Bayesian workflow:

1. Define a generative model matched to the outcome scale.
2. Check priors before fitting.
3. Fit with NUTS and inspect diagnostics.
4. Check posterior predictive fit.
5. Compare scientifically motivated model extensions with PSIS-LOO.
6. Translate posterior uncertainty into decision-relevant probabilities.

Important limitation: the data are synthetic. The numerical conclusions below should not be interpreted as clinical evidence about Long COVID.

In [ ]:
# If running in Google Colab, install the required packages.
# This is safe to re-run; Colab may ask you to restart only after major dependency changes.
%pip install -q pymc arviz pandas seaborn scipy

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

az.style.use("arviz-whitegrid")
sns.set_context("notebook")

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## 1. Scientific Setup

Symptom duration is strictly positive and typically right-skewed: many patients recover within a few months, while some have much longer durations. A Normal likelihood on raw months is therefore scientifically inappropriate because it assigns probability to negative durations.

We model durations with a LogNormal likelihood:

$$
y_i \sim \text{LogNormal}(\eta_i, \sigma),
$$

where $\eta_i$ is the expected log-duration for patient $i$, and $\exp(\eta_i)$ is the median duration for patients with those predictors.

In [ ]:
# Synthetic cohort design. These values are chosen to be plausible for an educational example,
# not to represent real epidemiological estimates.
n_patients = 160
hospitals = np.array(["Hospital A", "Hospital B", "Hospital C", "Hospital D"])
hospital_idx = rng.choice(len(hospitals), size=n_patients, p=[0.35, 0.30, 0.25, 0.10])

age = np.clip(rng.normal(loc=52, scale=15, size=n_patients), 18, 85)
age_z = (age - age.mean()) / age.std(ddof=0)

# True data-generating parameters on the log-months scale.
true_alpha = np.log(3.0)          # median duration around 3 months for a reference patient
true_beta_age = 0.22             # per 1 SD older age: exp(0.22) ~= 1.25x median duration
true_hospital_offset = np.array([-0.10, 0.00, 0.15, 0.35])
true_sigma = 0.55

eta = true_alpha + true_beta_age * age_z + true_hospital_offset[hospital_idx]
duration_months = rng.lognormal(mean=eta, sigma=true_sigma)

df = pd.DataFrame({
    "duration_months": duration_months,
    "log_duration": np.log(duration_months),
    "age": age,
    "age_z": age_z,
    "hospital": hospitals[hospital_idx],
    "hospital_idx": hospital_idx,
})

df.head()

In [ ]:
summary = df["duration_months"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.9, 0.95])
print(summary.round(2))

by_hospital = (
    df.groupby("hospital")
      .agg(n=("duration_months", "size"),
           median_months=("duration_months", "median"),
           mean_months=("duration_months", "mean"),
           p90_months=("duration_months", lambda x: np.percentile(x, 90)))
      .round(2)
)
display(by_hospital)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["duration_months"], bins=30, kde=True, ax=axes[0], color="steelblue")
axes[0].set(title="Raw symptom duration", xlabel="Months")
sns.histplot(df["log_duration"], bins=30, kde=True, ax=axes[1], color="darkorange")
axes[1].set(title="Log symptom duration", xlabel="log(months)")
plt.tight_layout()

## 2. Baseline LogNormal Model

The baseline model estimates one population-level median duration. This is intentionally simple and useful as a reference model.

Priors are weakly informative and expressed on clinically interpretable scales:

- $\alpha \sim \mathcal{N}(\log 3, 0.7)$ says the prior median is centered near 3 months, but allows wide uncertainty.
- $\sigma \sim \text{HalfNormal}(0.7)$ keeps the log-scale residual standard deviation positive and regularized.

In [ ]:
coords_base = {"obs_id": np.arange(len(df))}
y = df["duration_months"].to_numpy()

with pm.Model(coords=coords_base) as baseline_model:
    alpha = pm.Normal("alpha", mu=np.log(3.0), sigma=0.7)
    sigma = pm.HalfNormal("sigma", sigma=0.7)

    duration = pm.LogNormal("duration", mu=alpha, sigma=sigma, observed=y, dims="obs_id")
    median_duration = pm.Deterministic("median_duration", pm.math.exp(alpha))

    prior_base = pm.sample_prior_predictive(samples=1000, random_seed=RANDOM_SEED)

prior_draws = prior_base.prior_predictive["duration"].values.reshape(-1)
print("Prior predictive quantiles (months):", np.quantile(prior_draws, [0.01, 0.05, 0.5, 0.95, 0.99]).round(2))
print(f"P(prior predictive duration < 0) = {(prior_draws < 0).mean():.3f}")
print(f"P(prior predictive duration > 48 months) = {(prior_draws > 48).mean():.3f}")

az.plot_ppc(prior_base, group="prior", num_pp_samples=100, figsize=(8, 4))
plt.title("Prior predictive check: baseline LogNormal model")
plt.xlim(0, 60)
plt.show()

In [ ]:
with baseline_model:
    idata_base = pm.sample(
        draws=1500,
        tune=1000,
        chains=4,
        target_accept=0.90,
        random_seed=RANDOM_SEED,
        idata_kwargs={"log_likelihood": True},
    )

In [ ]:
def diagnostic_summary(idata, var_names):
    summary = az.summary(idata, var_names=var_names, hdi_prob=0.94)
    display(summary)

    max_rhat = float(summary["r_hat"].max())
    min_ess_bulk = float(summary["ess_bulk"].min())
    min_ess_tail = float(summary["ess_tail"].min())
    divergences = int(idata.sample_stats["diverging"].sum())

    print(f"Max R-hat: {max_rhat:.3f}")
    print(f"Minimum bulk ESS: {min_ess_bulk:.0f}")
    print(f"Minimum tail ESS: {min_ess_tail:.0f}")
    print(f"Divergences: {divergences}")

    if max_rhat > 1.01:
        print("WARNING: R-hat is above 1.01. Do not interpret the posterior until sampling is improved.")
    if min_ess_bulk < 400 or min_ess_tail < 400:
        print("WARNING: ESS is low. Increase draws or reparameterize before reporting.")
    if divergences > 0:
        print("WARNING: Divergences detected. Increase target_accept and inspect model geometry.")

    return summary

summary_base = diagnostic_summary(idata_base, ["alpha", "sigma", "median_duration"])
az.plot_trace(idata_base, var_names=["alpha", "sigma", "median_duration"]);
plt.tight_layout()

In [ ]:
posterior_median = idata_base.posterior["median_duration"].values.reshape(-1)
print(f"Posterior mean median duration: {posterior_median.mean():.2f} months")
print("94% HDI:", az.hdi(posterior_median, hdi_prob=0.94).round(2))
print(f"P(population median > 3 months | data, model) = {(posterior_median > 3).mean():.1%}")
print(f"P(population median > 6 months | data, model) = {(posterior_median > 6).mean():.1%}")

az.plot_posterior(idata_base, var_names=["median_duration"], hdi_prob=0.94, ref_val=3.0)
plt.title("Posterior for population median duration")
plt.show()

## 3. Posterior Predictive Checks

A model can have clean MCMC diagnostics and still be scientifically poor. Posterior predictive checks ask whether data simulated from the fitted model resemble the observed data. Here we check both the whole distribution and clinically relevant tail behavior.

In [ ]:
with baseline_model:
    idata_base = pm.sample_posterior_predictive(
        idata_base,
        var_names=["duration"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
az.plot_ppc(idata_base, group="posterior", num_pp_samples=100, ax=axes[0])
axes[0].set_title("Posterior predictive overlay")
axes[0].set_xlim(0, 30)

ppc_draws = idata_base.posterior_predictive["duration"].values.reshape(-1, len(df))
ppc_p90 = np.percentile(ppc_draws, 90, axis=1)
obs_p90 = np.percentile(y, 90)
axes[1].hist(ppc_p90, bins=40, color="steelblue", alpha=0.65, density=True)
axes[1].axvline(obs_p90, color="firebrick", lw=2, label=f"Observed p90 = {obs_p90:.1f} months")
axes[1].set(title="PPC test statistic: 90th percentile", xlabel="Months")
axes[1].legend()
plt.tight_layout()

print(f"Bayesian p-value for 90th percentile = {(ppc_p90 > obs_p90).mean():.2f}")
print(f"Observed fraction > 6 months = {(y > 6).mean():.1%}")
print(f"Posterior predictive fraction > 6 months = {(ppc_draws > 6).mean():.1%}")

## 4. Add an Age Predictor

The baseline model ignores patient-level heterogeneity. A scientifically motivated extension is to allow the median duration to vary with standardized age:

$$
\eta_i = \alpha + \beta_{age} \cdot age_{z,i}.
$$

The exponentiated coefficient $\exp(\beta_{age})$ is the multiplicative change in median duration for a one-standard-deviation increase in age.

In [ ]:
coords_age = {"obs_id": np.arange(len(df))}

with pm.Model(coords=coords_age) as age_model:
    age_data = pm.Data("age_z", df["age_z"].to_numpy(), dims="obs_id")

    alpha = pm.Normal("alpha", mu=np.log(3.0), sigma=0.7)
    beta_age = pm.Normal("beta_age", mu=0.0, sigma=0.35)
    sigma = pm.HalfNormal("sigma", sigma=0.7)

    eta = alpha + beta_age * age_data
    duration = pm.LogNormal("duration", mu=eta, sigma=sigma, observed=y, dims="obs_id")

    median_reference = pm.Deterministic("median_reference", pm.math.exp(alpha))
    median_ratio_per_age_sd = pm.Deterministic("median_ratio_per_age_sd", pm.math.exp(beta_age))

    prior_age = pm.sample_prior_predictive(samples=500, random_seed=RANDOM_SEED)
    idata_age = pm.sample(
        draws=1500,
        tune=1000,
        chains=4,
        target_accept=0.90,
        random_seed=RANDOM_SEED,
        idata_kwargs={"log_likelihood": True},
    )
    idata_age = pm.sample_posterior_predictive(
        idata_age, var_names=["duration"], extend_inferencedata=True, random_seed=RANDOM_SEED
    )

summary_age = diagnostic_summary(idata_age, ["alpha", "beta_age", "sigma", "median_ratio_per_age_sd"])
az.plot_posterior(idata_age, var_names=["median_ratio_per_age_sd"], hdi_prob=0.94, ref_val=1.0)
plt.title("Age effect as multiplicative median ratio")
plt.show()

## 5. Hierarchical Hospital Model

Patients are nested within hospitals. A hierarchical model uses partial pooling: hospitals with few observations borrow strength from the population, while hospitals with more data can deviate more strongly.

We use a non-centered parameterization for the hospital offsets, which is usually more stable for hierarchical scale parameters.

In [ ]:
coords_hier = {"obs_id": np.arange(len(df)), "hospital": hospitals}

with pm.Model(coords=coords_hier) as hierarchical_model:
    age_data = pm.Data("age_z", df["age_z"].to_numpy(), dims="obs_id")
    hospital_data = pm.Data("hospital_idx", df["hospital_idx"].to_numpy(), dims="obs_id")

    alpha = pm.Normal("alpha", mu=np.log(3.0), sigma=0.7)
    beta_age = pm.Normal("beta_age", mu=0.0, sigma=0.35)

    tau_hospital = pm.HalfNormal("tau_hospital", sigma=0.35)
    z_hospital = pm.Normal("z_hospital", mu=0.0, sigma=1.0, dims="hospital")
    hospital_offset = pm.Deterministic("hospital_offset", z_hospital * tau_hospital, dims="hospital")

    sigma = pm.HalfNormal("sigma", sigma=0.7)
    eta = alpha + beta_age * age_data + hospital_offset[hospital_data]

    duration = pm.LogNormal("duration", mu=eta, sigma=sigma, observed=y, dims="obs_id")
    median_reference = pm.Deterministic("median_reference", pm.math.exp(alpha))
    median_ratio_per_age_sd = pm.Deterministic("median_ratio_per_age_sd", pm.math.exp(beta_age))
    hospital_median_reference = pm.Deterministic(
        "hospital_median_reference", pm.math.exp(alpha + hospital_offset), dims="hospital"
    )

    idata_hier = pm.sample(
        draws=2000,
        tune=1500,
        chains=4,
        target_accept=0.95,
        random_seed=RANDOM_SEED,
        idata_kwargs={"log_likelihood": True},
    )
    idata_hier = pm.sample_posterior_predictive(
        idata_hier, var_names=["duration"], extend_inferencedata=True, random_seed=RANDOM_SEED
    )

summary_hier = diagnostic_summary(
    idata_hier,
    ["alpha", "beta_age", "tau_hospital", "sigma", "median_ratio_per_age_sd", "hospital_median_reference"],
)
az.plot_forest(idata_hier, var_names=["hospital_median_reference"], combined=True, hdi_prob=0.94)
plt.title("Partially pooled hospital-specific reference medians")
plt.show()

## 6. Model Comparison with PSIS-LOO

PSIS-LOO estimates expected out-of-sample predictive accuracy. On the log scale, higher ELPD-LOO is better. Differences should be interpreted with their standard errors; a tiny ELPD difference is not meaningful.

In [ ]:
comparison = az.compare(
    {
        "baseline": idata_base,
        "age": idata_age,
        "age + hospital": idata_hier,
    },
    ic="loo",
    scale="log",
)
display(comparison)

az.plot_compare(comparison)
plt.title("Model comparison by PSIS-LOO")
plt.show()

In [ ]:
loo_hier = az.loo(idata_hier, pointwise=True)
pareto_k = loo_hier.pareto_k.values

print(loo_hier)
print(f"Max Pareto-k: {np.nanmax(pareto_k):.2f}")
print(f"Number of observations with Pareto-k > 0.7: {(pareto_k > 0.7).sum()}")

fig, ax = plt.subplots(figsize=(10, 3))
colors = np.where(pareto_k > 0.7, "firebrick", np.where(pareto_k > 0.5, "darkorange", "steelblue"))
ax.scatter(np.arange(len(pareto_k)), pareto_k, c=colors, alpha=0.8)
ax.axhline(0.5, color="darkorange", ls="--", label="k = 0.5")
ax.axhline(0.7, color="firebrick", ls="--", label="k = 0.7")
ax.set(xlabel="Patient index", ylabel="Pareto-k", title="PSIS-LOO influence diagnostic")
ax.legend()
plt.tight_layout()

## 7. Predictive Calibration

LOO-PIT checks whether held-out observations look like draws from the model's predictive distribution. A roughly uniform LOO-PIT distribution is what we hope to see. U-shaped patterns suggest under-dispersed predictions; hump-shaped patterns suggest over-dispersed predictions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
az.plot_loo_pit(idata_hier, y="duration", ax=axes[0])
axes[0].set_title("LOO-PIT density")
az.plot_loo_pit(idata_hier, y="duration", ecdf=True, ax=axes[1])
axes[1].set_title("LOO-PIT ECDF")
plt.tight_layout()

## 8. Prior Sensitivity

A result is more trustworthy if the qualitative conclusion is stable under reasonable prior choices. Below we vary the prior width on the age coefficient while keeping the model structure fixed.

In [ ]:
sensitivity_idatas = {}
for beta_sigma in [0.20, 0.35, 0.60]:
    with pm.Model(coords=coords_age) as sensitivity_model:
        age_data = pm.Data("age_z", df["age_z"].to_numpy(), dims="obs_id")
        alpha = pm.Normal("alpha", mu=np.log(3.0), sigma=0.7)
        beta_age = pm.Normal("beta_age", mu=0.0, sigma=beta_sigma)
        sigma = pm.HalfNormal("sigma", sigma=0.7)
        eta = alpha + beta_age * age_data
        duration = pm.LogNormal("duration", mu=eta, sigma=sigma, observed=y, dims="obs_id")
        median_ratio_per_age_sd = pm.Deterministic("median_ratio_per_age_sd", pm.math.exp(beta_age))

        sensitivity_idatas[f"beta_sigma={beta_sigma}"] = pm.sample(
            draws=1000,
            tune=800,
            chains=2,
            target_accept=0.90,
            random_seed=RANDOM_SEED,
            progressbar=False,
        )

az.plot_forest(
    list(sensitivity_idatas.values()),
    model_names=list(sensitivity_idatas.keys()),
    var_names=["median_ratio_per_age_sd"],
    combined=True,
    hdi_prob=0.94,
)
plt.axvline(1.0, color="gray", ls="--")
plt.title("Prior sensitivity for the age effect")
plt.show()

## 9. Decision-Relevant Prediction

Posterior parameter estimates answer population questions. For a future patient's duration, use the posterior predictive distribution. Here we estimate the probability that a 70-year-old patient treated at Hospital D has symptoms longer than 6 or 12 months, under the fitted hierarchical model.

In [ ]:
posterior = idata_hier.posterior.stack(sample=("chain", "draw"))

new_age = 70
new_age_z = (new_age - df["age"].mean()) / df["age"].std(ddof=0)
new_hospital = "Hospital D"

eta_new = (
    posterior["alpha"].values
    + posterior["beta_age"].values * new_age_z
    + posterior["hospital_offset"].sel(hospital=new_hospital).values
)
sigma_new = posterior["sigma"].values

rng_decision = np.random.default_rng(123)
pred_new = rng_decision.lognormal(mean=eta_new, sigma=sigma_new)

print(f"Prediction for age {new_age}, {new_hospital}")
print("Predictive median and 94% interval (months):", np.quantile(pred_new, [0.5, 0.03, 0.97]).round(2))
print(f"P(duration > 6 months | data, model) = {(pred_new > 6).mean():.1%}")
print(f"P(duration > 12 months | data, model) = {(pred_new > 12).mean():.1%}")

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(pred_new, bins=50, kde=True, ax=ax, color="slateblue")
ax.axvline(6, color="darkorange", ls="--", label="6 months")
ax.axvline(12, color="firebrick", ls="--", label="12 months")
ax.set(title="Posterior predictive duration for a future patient", xlabel="Months")
ax.legend()
plt.xlim(0, np.percentile(pred_new, 99))
plt.show()

## 10. Reporting Checklist

For a scientific report, include:

- The data-generating assumptions and the reason for the LogNormal likelihood.
- Prior predictive checks showing that simulated durations are plausible.
- MCMC diagnostics: R-hat, ESS, divergences, trace plots, and energy/BFMI.
- Posterior summaries with 94% credible intervals or HDIs.
- Posterior predictive checks on clinically meaningful summaries, not only visual overlays.
- PSIS-LOO comparison with standard errors and Pareto-k diagnostics.
- Sensitivity analysis for priors that could affect substantive conclusions.
- Decision probabilities based on posterior predictive draws when predicting future patient outcomes.

References and API notes:

- PyMC posterior predictive API: https://www.pymc.io/projects/docs/en/stable/api/generated/pymc.sample_posterior_predictive.html
- PyMC log-likelihood API: https://www.pymc.io/projects/docs/en/latest/api/generated/pymc.compute_log_likelihood.html
- ArviZ LOO-PIT API: https://python.arviz.org/en/stable/api/generated/arviz.plot_loo_pit.html
- Gelman et al., Bayesian Workflow: https://arxiv.org/abs/2011.01808